# 33 - Bias Mitigation Strategies - Lab

## Description

This lab implements two mitigation techniques for reducing bias in classification tasks. Starting from a biased lending classifier trained on the Adult Income dataset (loaded from **Lesson 32**), students apply re-weighing and adversarial debiasing, measure fairness and accuracy before and after each technique, and compare the accuracy-fairness trade-offs in a single summary table.

## Objectives

- Implement re-weighing using AIF360 and explain how the computed weights correct the skewed joint distribution of protected attribute and label.
- Implement adversarial debiasing using AIF360, explain the two-model training loop, and describe what the adversary's prediction loss signals to the classifier.
- Produce a side-by-side comparison of re-weighing and adversarial debiasing showing accuracy cost and fairness improvement across two fairness metrics.

## Prerequisites

This lesson's lecture content, plus lesson 32 (Fairness and Bias in AI Systems). Students must understand Demographic Parity, Equalized Odds, and how fairness metrics are computed using AIF360.

## Required Software/Packages

- AIF360 (Algorithmic Fairness 360): `pip install aif360`
- scikit-learn (version 1.0+): included in standard environment
- pandas, numpy, matplotlib: included in standard environment
- TensorFlow/Keras (for adversarial debiasing): included in standard environment
- tf-keras (Keras 2 compatibility layer): `pip install tf-keras`
- pandas < 3.0 (aif360 0.6.x is not yet compatible with pandas 3)

**Dataset download**: `AdultDataset` expects the raw UCI Adult files (`adult.data`, `adult.test`, `adult.names`) to already exist inside your `aif360` installation. A fresh `aif360` install (as on a new Colab runtime) does not ship with these files, which is why the dataset-loading cell can otherwise fail with a "No such file or directory" error. The dataset-download cell further below fixes this automatically by downloading the three files straight into the folder `AdultDataset` expects, so you no longer need to do this by hand.

## Environment/Setup Instructions

Run the cell below to make sure `aif360` is installed, then run the import cell to load all required libraries. This lab uses the Adult Income dataset, downloaded automatically into AIF360's expected data folder by the dataset-download cell below.

In [ ]:
# Install aif360 if it is not already available (needed on a fresh Colab runtime)
try:
    import aif360
except ImportError:
    !pip install aif360
    import aif360

print("aif360 is available.")

In [ ]:
# Import required libraries
import os
# aif360's AdversarialDebiasing requires the Keras 2 API bundled with tf-keras
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow.compat.v1 as tf
tf.disable_eager_execution()
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from aif360.datasets import AdultDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing import AdversarialDebiasing
from aif360.metrics import BinaryLabelDatasetMetric
import warnings
warnings.filterwarnings('ignore')

## Background/Context

Bias mitigation techniques operate at different points in the ML pipeline. This lab compares a pre-processing method (re-weighing) and an in-processing method (adversarial debiasing) on a lending classification task. The Adult Income dataset includes gender as a protected attribute; we measure whether male and female applicants receive equal positive-prediction rates (Demographic Parity) before and after mitigation. Accuracy is measured on the original (unweighted, undebiased) test set.

## Exercise Instructions

1. Load the Adult Income dataset from AIF360, split into train/test, train a baseline logistic regression classifier, and compute baseline Demographic Parity difference and accuracy.
2. Apply Reweighing to training data, retrain the logistic regression classifier on reweighted samples, and compute Demographic Parity difference and accuracy on the test set. Record results. 
3. Apply AdversarialDebiasing with the default two-model architecture, train the debiased classifier, and compute Demographic Parity difference and accuracy on the test set. Record results. 
4. Create a comparison table with rows for baseline, re-weighted, and adversarial techniques and columns for DP Difference Before, DP Difference After, Accuracy Before, and Accuracy After. Write one sentence interpreting which technique offers the better fairness-accuracy trade-off. 

## Starter Code

### Download the raw Adult dataset files

`AdultDataset` expects `adult.data`, `adult.test`, and `adult.names` to already exist inside the `aif360` package's `data/raw/adult/` folder. A fresh `aif360` install (as on a new Colab runtime) does not ship with these files, which is why the dataset-loading cell below can otherwise fail with a "No such file or directory" error. This cell downloads the three files directly from the UCI Machine Learning Repository and places them exactly where `AdultDataset` expects them, so the loading cell that follows just works.

In [ ]:
import os
import urllib.request

# Define the target directory expected by AIF360
target_dir = "/usr/local/lib/python3.13/dist-packages/aif360/data/raw/adult"
os.makedirs(target_dir, exist_ok=True)

# URLs for the UCI Adult dataset files
urls = {
    "adult.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "adult.test": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
    "adult.names": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names"
}

# Download each file and place it in the correct directory
for filename, url in urls.items():
    filepath = os.path.join(target_dir, filename)
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"Saved to {filepath}")
    else:
        print(f"{filename} already exists.")

print("Dataset download and structure setup complete!")

In [ ]:
# Load and prepare the Adult Income dataset
dataset_original = AdultDataset(
    label_name='income-per-year',
    favorable_classes=['>50K', '>50K.'],
    protected_attribute_names=['sex'],
    privileged_classes=[['Male']],  # Male is the privileged group
    categorical_features=['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country'],
    features_to_keep=['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', 'sex', 'workclass', 'occupation', 'marital-status', 'race'],
    features_to_drop=[]
)

# Split into train and test (70-30 split)
np.random.seed(42)
n_samples = dataset_original.features.shape[0]
train_size = int(0.7 * n_samples)
indices = np.random.permutation(n_samples)
train_indices = indices[:train_size]
test_indices = indices[train_size:]

dataset_train = dataset_original.subset(train_indices.tolist())
dataset_test = dataset_original.subset(test_indices.tolist())

print(f"Training set size: {dataset_train.features.shape[0]}")
print(f"Test set size: {dataset_test.features.shape[0]}")
print(f"Protected attribute: {dataset_original.protected_attribute_names}")
print(f"Target variable: {dataset_original.label_names}")

### Visualize the Data

Before training anything, look at how `sex` and `race` are distributed in the training data. The pie charts below make the class imbalance in these protected attributes concrete: when one group vastly outnumbers another, a classifier trained on this data has far less signal to learn accurate predictions for the minority group — this is part of why the baseline classifier you train next is biased in the first place.

In [ ]:
# Decode the protected/categorical attributes back to readable labels for plotting
train, _ = dataset_train.convert_to_dataframe(de_dummy_code=True)

# Extract the actual raw binary values for 'sex' directly from the protected attributes
sex_index = dataset_train.protected_attribute_names.index('sex')
raw_sex_vals = dataset_train.protected_attributes[:, sex_index]

# Create a Series with correct mapping
gender_series = pd.Series(raw_sex_vals).map({1.0: 'Male', 0.0: 'Female'})

# Calculate the distribution of sex and race
gender_counts = gender_series.value_counts()
gender_labels = gender_counts.index
gender_sizes = gender_counts.values

race_counts = train['race'].value_counts()
race_labels = race_counts.index
race_sizes = race_counts.values

plt.figure(figsize=(10, 5))

# Subplot 1: Gender Distribution
plt.subplot(1, 2, 1)
plt.pie(gender_sizes, startangle=70)
plt.title('Gender Distribution')
plt.legend(gender_labels, loc='upper right')

# Subplot 2: Race Distribution
plt.subplot(1, 2, 2)
plt.pie(race_sizes, startangle=0)
plt.title('Race Distribution')
plt.legend(race_labels, loc='upper right', bbox_to_anchor=(1.3, 1))

plt.tight_layout()
plt.show()

## Step 1: Baseline Classifier 

In [ ]:
# Train baseline logistic regression classifier
X_train = dataset_train.features
y_train = dataset_train.labels.ravel()

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train baseline model
baseline_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)

# Make predictions on test set
X_test = dataset_test.features
X_test_scaled = scaler.transform(X_test)
y_test = dataset_test.labels.ravel()

# Create prediction dataset for fairness metrics
dataset_test_pred = dataset_test.copy()
dataset_test_pred.labels = baseline_model.predict(X_test_scaled).reshape(-1, 1)

# Compute baseline fairness metrics
metric_baseline = BinaryLabelDatasetMetric(
    dataset_test_pred,
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

dp_diff_baseline = abs(metric_baseline.mean_difference())
accuracy_baseline = np.mean(y_test == baseline_model.predict(X_test_scaled))

print(f"Baseline Demographic Parity Difference: {dp_diff_baseline:.4f}")
print(f"Baseline Accuracy: {accuracy_baseline:.4f}")

## Fairness Metrics

These metrics quantify fairness in a model's predictions. They compare how outcomes differ between **privileged** (e.g., Male, White) and **unprivileged** (e.g., Female, Non-White) groups in the **Adult dataset**. The numeric "Interpretation" values below are illustrative examples from a different run, not this lab's own results — use them to learn how to read the metrics, then interpret your own `dp_diff_baseline`, `dp_diff_reweighted`, and `dp_diff_adversarial` values the same way.

Before diving in, here are some important classification terms that appear in the formulas:

**Confusion Matrix**

<div style="display: flex;">
  <span style="margin-right: 10px;">
    <img src="https://miro.medium.com/v2/resize:fit:640/format:webp/1*-hnHerS0oitp3X0yEjgCHw.png" alt="Confusion matrix" width="300">
  </span>
  <span>
    <img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*TAgP-j_02dPpfRiL0fbP-Q.png" alt="Confusion matrix formulas" width="350">
  </span>
</div>

- **True Positives (TP):** Cases where the model correctly predicts high income (>50K).
- **False Positives (FP):** Cases where the model predicts high income, but the person actually earns ≤50K.
- **True Negatives (TN):** Cases where the model correctly predicts ≤50K.
- **False Negatives (FN):** Cases where the model predicts ≤50K, but the person actually earns >50K.

From these:

- **Sensitivity (True Positive Rate, TPR):**
  $$
  \text{TPR} = \frac{TP}{TP + FN}
  $$
  Proportion of actual high-income individuals correctly identified.

- **Specificity (True Negative Rate, TNR):**
  $$
  \text{TNR} = \frac{TN}{TN + FP}
  $$
  Proportion of actual low-income individuals correctly identified.

- **False Positive Rate (FPR):**
  $$
  \text{FPR} = \frac{FP}{FP + TN}
  $$
  Proportion of low-income individuals wrongly labeled as high income.

---

**Balanced Accuracy**

- **Definition**: Average of sensitivity (TPR) and specificity (TNR). Helps when classes are imbalanced (the Adult dataset has fewer >50K earners).

  $$
  \text{Balanced Accuracy} = \frac{\text{Sensitivity} + \text{Specificity}}{2}
  $$

- **Interpretation (example)**: A value of **0.7437** means the model is reasonably accurate across both income groups, not just the majority (≤50K).

---

**Statistical Parity Difference (SPD)** — this is what this lab's `dp_diff_baseline` / `dp_diff_reweighted` / `dp_diff_adversarial` (`mean_difference()`) compute below.

- **Definition**: Difference in probability of receiving a favorable prediction between unprivileged and privileged groups.

  $$
  \text{SPD} = P(\hat{Y}=1 \mid \text{unprivileged}) - P(\hat{Y}=1 \mid \text{privileged})
  $$

- **$P(\hat{Y}=1 \mid \text{unprivileged})$**: Probability that the model predicts a *favorable outcome* (e.g., income >50K) for someone in the unprivileged group.
- **$P(\hat{Y}=1 \mid \text{privileged})$**: Same probability, but for the privileged group.

- **SPD = 0** → Both groups treated equally.
- **SPD < 0** → Unprivileged group is *less likely* to get the favorable outcome.
- **SPD > 0** → Unprivileged group is *more likely*.

- **Interpretation (example)**: **-0.3580** → unprivileged group is **35.8% less likely** to be predicted as high income.
  - **Ideal Value**: 0 (both groups equally likely). This lab reports `abs(mean_difference())`, so a value near 0 is the goal regardless of sign.

---

**Disparate Impact (DI)**

- **Definition**: Ratio of favorable outcome rates between unprivileged and privileged groups.

  $$
  \text{DI} = \frac{P(\hat{Y}=1 \mid \text{unprivileged})}{P(\hat{Y}=1 \mid \text{privileged})}
  $$

This uses the same probabilities as SPD but forms a **ratio** instead of a difference.

- **DI = 1** → Perfect fairness (equal rates).
- **DI < 1** → Unprivileged group gets fewer favorable outcomes.
- **DI > 1** → Unprivileged group gets more favorable outcomes.

- **Interpretation (example)**: **0.2794** → unprivileged group is only **27.94% as likely** to get a positive prediction.
  - **Ideal Value**: 1 (perfect fairness).
  - Below 0.8 → often flagged as unfair (the "80% rule" / "four-fifths rule").

---

**Average Odds Difference (AOD)**

- **Definition**: Average of the differences in **false positive rates (FPR)** and **true positive rates (TPR)** between groups.

  $$
  \text{AOD} = \frac{(\text{FPR}_{\text{unprivileged}} - \text{FPR}_{\text{privileged}}) + (\text{TPR}_{\text{unprivileged}} - \text{TPR}_{\text{privileged}})}{2}
  $$

- **Interpretation (example)**: **-0.3181** → unprivileged individuals are both less likely to be correctly identified when they truly earn >50K (lower TPR) **and/or** more likely to be misclassified when they don't (higher FPR).
  - **Ideal Value**: 0 (both groups have equal error behavior).

---

**Equal Opportunity Difference (EOD)** — this is the metric Optional Challenge 1 below asks you to add.

- **Definition**: Difference in **true positive rates (TPR)** between unprivileged and privileged groups.

  $$
  \text{EOD} = \text{TPR}_{\text{unprivileged}} - \text{TPR}_{\text{privileged}}
  $$

- **Interpretation (example)**: **-0.3769** → unprivileged individuals who truly earn >50K are **37.69% less likely** to be correctly recognized.
  - Focuses only on fairness in *positive outcomes*.
  - **Ideal Value**: 0.

---

**Theil Index**

- **Definition**: A general inequality measure adapted from economics. It looks at the distribution of predicted outcomes across individuals — if some groups are disproportionately burdened with errors, the index rises. Related to entropy: higher values mean more inequality.

- **Interpretation (example)**: **0.1129** → there is measurable inequality in the predictions, even beyond just group-level fairness.
  - **Ideal Value**: 0.

---

**Summary**
- **Balanced Accuracy** → overall model reliability (performance metric).
- **SPD, DI, AOD, EOD** → compare fairness across privileged vs unprivileged groups.
- **Theil Index** → captures broader inequality across the dataset.

## Expected Output: Step 1

A Demographic Parity Difference value between 0 and 1 (where 0 = perfect parity, 1 = maximum disparity) and a baseline accuracy value between 0 and 1.

## Step 2: Re-weighing Mitigation

In [ ]:
# Apply Reweighing to the training data
reweighing = Reweighing(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

# Fit and transform the training data
dataset_train_reweighted = reweighing.fit_transform(dataset_train)

# Extract reweighted training data
X_train_reweighted = dataset_train_reweighted.features
y_train_reweighted = dataset_train_reweighted.labels.ravel()
sample_weights = dataset_train_reweighted.instance_weights.ravel()

# Standardize reweighted features
X_train_reweighted_scaled = scaler.fit_transform(X_train_reweighted)

# Train logistic regression on reweighted data
reweighted_model = LogisticRegression(random_state=42, max_iter=1000)
reweighted_model.fit(X_train_reweighted_scaled, y_train_reweighted, sample_weight=sample_weights)

# Make predictions on test set
dataset_test_reweighted_pred = dataset_test.copy()
dataset_test_reweighted_pred.labels = reweighted_model.predict(X_test_scaled).reshape(-1, 1)

# Compute fairness metrics for reweighted model
metric_reweighted = BinaryLabelDatasetMetric(
    dataset_test_reweighted_pred,
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

dp_diff_reweighted = abs(metric_reweighted.mean_difference())
accuracy_reweighted = np.mean(y_test == reweighted_model.predict(X_test_scaled))

print(f"Re-weighted Demographic Parity Difference: {dp_diff_reweighted:.4f}")
print(f"Re-weighted Accuracy: {accuracy_reweighted:.4f}")
print(f"Accuracy Drop from Baseline: {accuracy_baseline - accuracy_reweighted:.4f}")
print(f"Fairness Improvement (DP Diff Reduction): {dp_diff_baseline - dp_diff_reweighted:.4f}")

### How Reweighing computes its weights

`Reweighing.fit_transform` is what produced the `sample_weights` used above. For every combination of protected-attribute group $A$ (privileged/unprivileged, here Male/Female) and label $Y$ (favorable/unfavorable, here income >50K or not), it assigns one weight (Kamiran and Calders, 2012):

$$
w(A, Y) = \frac{P(A) \, P(Y)}{P(A, Y)}
$$

- $P(A)$ — the fraction of the training set belonging to group $A$ (ignoring the label).
- $P(Y)$ — the fraction of the training set with outcome $Y$ (ignoring the group).
- $P(A, Y)$ — the fraction of the training set that is actually *both* in group $A$ *and* has outcome $Y$.

$P(A) \, P(Y)$ is what the joint probability *would* be if group membership and outcome were statistically independent — i.e., if income were unrelated to gender. $P(A, Y)$ is what the joint probability *actually* is in the (biased) training data. The weight is the ratio of the two:

- If a group-outcome combination occurs **less often than independence would predict** ($P(A,Y) < P(A)P(Y)$, e.g. few favorable outcomes for the unprivileged group), the ratio is **greater than 1**, and those examples get **upweighted**.
- If a combination occurs **more often than independence would predict** ($P(A,Y) > P(A)P(Y)$, e.g. favorable outcomes concentrated in the privileged group), the ratio is **less than 1**, and those examples get **downweighted**.
- If the combination occurs at exactly the rate independence predicts, $w(A,Y) = 1$: no correction needed.

**Worked example.** Suppose 70% of a training set is Male, 30% Female, and overall 30% of examples are favorable (income >50K). Under independence, Male-and-favorable should be $0.70 \times 0.30 = 0.21$ of the data. If it is actually $0.25$ (males are overrepresented among favorable outcomes), the weight is $0.21 / 0.25 = 0.84$ — those examples count less. Female-and-favorable should also be $0.30 \times 0.30 = 0.09$ of the data; if it is actually only $0.05$ (females are underrepresented among favorable outcomes), the weight is $0.09 / 0.05 = 1.8$ — those examples count almost twice as much.

**What this does *not* do:** Reweighing never adds, removes, or duplicates a single training example. `dataset_train_reweighted` has exactly the same rows as `dataset_train`. All it changes is `instance_weights`, the per-row weight each example carries into training. Passing those weights as `sample_weight` to `LogisticRegression.fit` (as the cell above does) changes how much each example's error contributes to the loss function the classifier minimizes — underrepresented, disadvantaged-group-with-favorable-outcome examples pull the decision boundary harder toward predicting "favorable" for that group, while overrepresented combinations pull less, without the training set's size or any individual row ever changing. This is why Reweighing is a **pre-processing** mitigation: it only touches the data (via these weights) before a completely standard classifier is trained on it.

**Reference:** Kamiran, F., and Calders, T. (2012). "Data preprocessing techniques for classification without discrimination." *Knowledge and Information Systems*, 33(1), 1-33. <https://doi.org/10.1007/s10115-011-0463-8> — the original paper defining this reweighing formula. See also the [AIF360 `Reweighing` API docs](https://aif360.readthedocs.io/en/latest/modules/generated/aif360.algorithms.preprocessing.Reweighing.html) for the implementation used in this lab.

## Expected Output: Step 2

The re-weighted Demographic Parity Difference (should be less than baseline if mitigation worked), the re-weighted accuracy, and the improvement/cost metrics.

## Step 3: Adversarial Debiasing 

In [ ]:
# Apply AdversarialDebiasing
sess = tf.Session()
adversarial_debiaser = AdversarialDebiasing(
    privileged_groups=[{'sex': 1}],
    unprivileged_groups=[{'sex': 0}],
    scope_name='debiased_classifier',
    sess=sess,
    num_epochs=10,
    seed=42
)

# Fit the debiased model on training data
adversarial_debiaser.fit(dataset_train)

# Make predictions on test set
dataset_test_adversarial_pred = adversarial_debiaser.predict(dataset_test)

# Compute fairness metrics for adversarial debiased model
metric_adversarial = BinaryLabelDatasetMetric(
    dataset_test_adversarial_pred,
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)

dp_diff_adversarial = abs(metric_adversarial.mean_difference())
accuracy_adversarial = np.mean(y_test == dataset_test_adversarial_pred.labels.ravel())

print(f"Adversarial Debiased Demographic Parity Difference: {dp_diff_adversarial:.4f}")
print(f"Adversarial Debiased Accuracy: {accuracy_adversarial:.4f}")
print(f"Accuracy Drop from Baseline: {accuracy_baseline - accuracy_adversarial:.4f}")
print(f"Fairness Improvement (DP Diff Reduction): {dp_diff_baseline - dp_diff_adversarial:.4f}")

sess.close()

## Expected Output: Step 3

The adversarially debiased Demographic Parity Difference (should be less than baseline if mitigation worked), the accuracy, and the improvement/cost metrics.

## Step 4: Comparison Table and Interpretation

In [ ]:
# Create comparison table
comparison_data = {
    'Technique': ['Baseline', 'Re-weighted', 'Adversarial Debiasing'],
    'DP Diff (Before)': [dp_diff_baseline, dp_diff_baseline, dp_diff_baseline],
    'DP Diff (After)': [dp_diff_baseline, dp_diff_reweighted, dp_diff_adversarial],
    'Accuracy (Before)': [accuracy_baseline, accuracy_baseline, accuracy_baseline],
    'Accuracy (After)': [accuracy_baseline, accuracy_reweighted, accuracy_adversarial]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nMitigation Techniques Comparison")
print(comparison_df.to_string(index=False))

# Compute and display trade-off metrics
print("\nTrade-off Analysis")
reweighted_fairness_gain = dp_diff_baseline - dp_diff_reweighted
reweighted_accuracy_loss = accuracy_baseline - accuracy_reweighted
adversarial_fairness_gain = dp_diff_baseline - dp_diff_adversarial
adversarial_accuracy_loss = accuracy_baseline - accuracy_adversarial

print(f"Re-weighing: Fairness gain = {reweighted_fairness_gain:.4f}, Accuracy loss = {reweighted_accuracy_loss:.4f}")
print(f"Adversarial: Fairness gain = {adversarial_fairness_gain:.4f}, Accuracy loss = {adversarial_accuracy_loss:.4f}")

## Expected Output: Step 4

A table with three rows and six columns showing DP Difference and Accuracy before and after mitigation for each technique. Below the table, fairness gain and accuracy loss for each mitigation technique.

## Questions/Tasks

1. In the re-weighing step, which group (privileged or unprivileged) received higher sample weights, and why does this make sense for correcting the Demographic Parity difference?
2. In adversarial debiasing, what role does the adversary play, and how does maximizing the adversary's loss signal guide the classifier toward learning fairer representations?

## Optional Challenges

**Challenge 1:** Compute and report Equalized Odds difference (equal TPR and FPR across groups) for each technique in addition to Demographic Parity. Which fairness metric improved more?

**Challenge 2:** Modify the hyperparameter λ (fairness-accuracy weight) in adversarial debiasing from 0.1 to 2.0 and plot how the Demographic Parity difference and accuracy change. What trade-off shape emerges?

## Assignment: Re-run with Modifications

Re-run this notebook (Kernel → Restart & Run All) once for each modification below, and record your results in a short report (a table plus a few sentences per part). For each run, report the baseline, re-weighted, and adversarial **DP difference (after)** and **accuracy (after)** values.

1. **Change the fairness–accuracy weight.** In Step 3, set `adversary_loss_weight=0.5` in the `AdversarialDebiasing` constructor (default is 0.1). Re-run and report the adversarial technique's DP difference and accuracy. Does the fairness gain justify the accuracy cost compared to λ = 0.1?

2. **Change the protected attribute to race.** In the dataset-loading cell, set `protected_attribute_names=['race']` and `privileged_classes=[['White']]`, and update every `{'sex': 0}` / `{'sex': 1}` group definition to `{'race': 0}` / `{'race': 1}`. Re-run the whole notebook. Which technique achieves the lower DP difference for race, and what accuracy does it cost?

3. **Change the train/test split.** In the loading cell, change the split from 70/30 to 50/50 (`int(0.5 * n_samples)`). Re-run. Do the fairness metrics change substantially? Briefly explain why a smaller training set might affect mitigation effectiveness.

